In [ ]:
# Cell 1: Import libraries

import os
import zipfile
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageOps

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
# Cell 2: Set paths

from pathlib import Path

PROJECT_ROOT = Path("..")

# Raw dataset
ZIP_PATH = PROJECT_ROOT / "data" / "raw" / "amharic_digit_dataset.zip"

# Extracted dataset
EXTRACT_DIR = PROJECT_ROOT / "data" / "extracted"

# Processed dataset
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Image size
IMAGE_SIZE = 128

# Folder number -> Ge'ez digit
CLASS_NAMES = {
    1: "፩",
    2: "፪",
    3: "፫",
    4: "፬",
    5: "፭",
    6: "፮",
    7: "፯",
    8: "፰",
    9: "፱",
}

print("ZIP path:", ZIP_PATH)
print("Extract directory:", EXTRACT_DIR)
print("Processed directory:", PROCESSED_DIR)

print("TensorFlow version:", tf.__version__)
print("Number of classes:", len(CLASS_NAMES))

In [ ]:
# Cell 3: Extract the ZIP file

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"ZIP file not found:\n{ZIP_PATH}\n\n"
        "Make sure the dataset ZIP is located inside data/raw/."
    )

# Remove previously extracted dataset
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

# Create extraction directory
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Extract the ZIP file
with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Dataset extracted successfully!")
print("Location:", EXTRACT_DIR.resolve())

In [ ]:
# Cell 4: Automatically find the directory containing classes 1-9

candidate_roots = []

for p in EXTRACT_DIR.rglob("*"):

    if not p.is_dir():
        continue

    # Get the names of subdirectories
    subfolders = {
        item.name
        for item in p.iterdir()
        if item.is_dir()
    }

    # Check whether this directory contains folders 1 through 9
    if all(str(i) in subfolders for i in range(1, 10)):
        candidate_roots.append(p)

if not candidate_roots:
    raise FileNotFoundError(
        f"Could not find class folders 1-9 inside:\n"
        f"{EXTRACT_DIR.resolve()}"
    )

# Use the first matching directory
DATASET_DIR = candidate_roots[0]

print("Dataset directory found:")
print(DATASET_DIR.resolve())
print()

# Count images in each class
image_extensions = {
    ".png", ".jpg", ".jpeg",
    ".bmp", ".tiff", ".webp"
}

for class_id, geez in CLASS_NAMES.items():

    folder = DATASET_DIR / str(class_id)

    images = [
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    ]

    print(f"Class {class_id} ({geez}): {len(images)} images")

In [ ]:
# Cell 5: Inspect image dimensions and color modes

records = []

# Supported image formats
image_extensions = {
    ".png", ".jpg", ".jpeg",
    ".bmp", ".tiff", ".webp"
}

for class_id in CLASS_NAMES:

    folder = DATASET_DIR / str(class_id)

    for path in folder.iterdir():

        # Ignore non-image files
        if not path.is_file() or path.suffix.lower() not in image_extensions:
            continue

        try:
            with Image.open(path) as img:

                records.append({
                    "class": class_id,
                    "geez": CLASS_NAMES[class_id],
                    "filename": path.name,
                    "width": img.width,
                    "height": img.height,
                    "mode": img.mode,
                    "format": img.format
                })

        except Exception as e:
            print("Could not read:", path, "|", e)


# Convert records into a DataFrame
df_info = pd.DataFrame(records)

# Basic information

print("Total images:", len(df_info))

print("\nFirst 5 images:")
display(df_info.head())

# Image modes

print("\nImage modes:")
print(df_info["mode"].value_counts())


# Image formats

print("\nImage formats:")
print(df_info["format"].value_counts())

# Image dimensions

print("\nImage dimension statistics:")
display(
    df_info[["width", "height"]].describe()
)


# Find maximum dimensions

max_width = int(df_info["width"].max())
max_height = int(df_info["height"].max())

print("\nMaximum Width :", max_width)
print("Maximum Height:", max_height)

In [ ]:
# Cell 6: Display sample images from all 9 classes

fig, axes = plt.subplots(3, 3, figsize=(9, 9))

for class_id, ax in zip(CLASS_NAMES, axes.ravel()):
    folder = DATASET_DIR / str(class_id)
    image_files = [p for p in folder.iterdir() if p.is_file()]

    if image_files:
            # Pick a random image file instead of always taking the first one
            random_image_path = random.choice(image_files)
            
            with Image.open(random_image_path) as img:
                ax.imshow(img)
                ax.set_title(f"{CLASS_NAMES[class_id]} (folder {class_id})")
                ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Preprocessing function

# Maximum dimensions identified from dataset inspection
MAX_WIDTH = 217
MAX_HEIGHT = 261

def preprocess_image(image_path, max_width=MAX_WIDTH, max_height=MAX_HEIGHT): # setting the size of the images
    # Open image
    img = Image.open(image_path) # loads the files into memory

    # If the image has transparency, place it on a white background first.
    if img.mode in ("RGBA", "LA"): # checks the transparacy of the images
        rgba = img.convert("RGBA") # made the image in rgba so that we can treat transparency channels
        background = Image.new("RGBA", rgba.size, "white") # creating a white background
        img = Image.alpha_composite(background, rgba).convert("L")
    else:
        img = img.convert("L")

    # Improve contrast
    img = ImageOps.autocontrast(img)

    # Keep aspect ratio while resizing
    img.thumbnail((max_width, max_height), Image.Resampling.LANCZOS)

    # Create a black canvas (filled with zeros) of size (MAX_HEIGHT, MAX_WIDTH)
    canvas_array = np.zeros((max_height, max_width), dtype=np.uint8)

    # Center the image on the zero-padded numpy canvas
    img_array = np.array(img)
    x = (max_width - img.width) // 2
    y = (max_height - img.height) // 2

    canvas_array[y:y + img.height, x:x + img.width] = img_array

    # Return as a PIL Image object compatible with Cell 8 and Cell 9
    return Image.fromarray(canvas_array)


In [ ]:
# Cell 8: Test preprocessing on one image sample
random_class_id = random.choice(list(CLASS_NAMES.keys()))
class_folder = DATASET_DIR / str(random_class_id)
image_files = [p for p in class_folder.iterdir() if p.is_file()]

sample_path = random.choice(image_files)
original = Image.open(sample_path)

# Call function using MAX_WIDTH and MAX_HEIGHT (217, 261)
processed = preprocess_image(sample_path, max_width=MAX_WIDTH, max_height=MAX_HEIGHT)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(original)
axes[0].set_title(f"Original ({CLASS_NAMES[random_class_id]})\n{original.size}, {original.mode}")
axes[0].axis("off")

axes[1].imshow(processed, cmap="gray")
axes[1].set_title(f"Processed\n{processed.size}, {processed.mode}")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 9: Process and save the complete dataset

if PROCESSED_DIR.exists():
    shutil.rmtree(PROCESSED_DIR)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

processed_count = 0

for class_id, geez in CLASS_NAMES.items():
    source_folder = DATASET_DIR / str(class_id)
    output_folder = PROCESSED_DIR / str(class_id)
    output_folder.mkdir(parents=True, exist_ok=True)

    image_number = 1

    for image_path in source_folder.iterdir():
        if not image_path.is_file():
            continue

        try:
            # Preprocess image to fit within MAX_WIDTH (217) and MAX_HEIGHT (261)
            processed = preprocess_image(image_path, max_width=MAX_WIDTH, max_height=MAX_HEIGHT)

            output_path = output_folder / f"{class_id}_{image_number:04d}.png"
            processed.save(output_path)

            processed_count += 1
            image_number += 1

        except Exception as e:
            print("Skipped:", image_path, "|", e)

print(f"Processed images: {processed_count}")
print("Saved to:", PROCESSED_DIR.resolve())

In [ ]:
# Cell 10: Verify processed images

# Set font family to display Ethiopic/Ge'ez characters properly on Windows
plt.rcParams['font.family'] = 'Segoe UI Historic'

fig, axes = plt.subplots(3, 3, figsize=(9, 9))

for class_id, ax in zip(CLASS_NAMES, axes.ravel()):
    folder = PROCESSED_DIR / str(class_id)
    image_files = list(folder.glob("*.png"))
    
    if image_files:
        # Pick a random image instead of always using index 0
        random_image_path = random.choice(image_files)
        
        with Image.open(random_image_path) as img:
            ax.imshow(img, cmap="gray")
            ax.set_title(f"{CLASS_NAMES[class_id]} — {img.size}")
            ax.axis("off")

plt.tight_layout()
plt.show()

print(f"All processed images should be {MAX_WIDTH} × {MAX_HEIGHT} grayscale PNG files.")

In [ ]:
# Cell 11: Load images into X and labels into y

X = [] # holding image pixel data
y = [] # holding the class labels

for class_id in CLASS_NAMES:
    folder = PROCESSED_DIR / str(class_id) # setting the pathes

    for image_path in sorted(folder.glob("*.png")): # going through every part
        with Image.open(image_path) as img:
            image_array = np.array(img, dtype=np.float32) #  opens the image 

        X.append(image_array) # adding the pixel data
        y.append(class_id - 1)  # Convert 1-9 to 0-8

X = np.array(X)
y = np.array(y)

print("X shape before normalization:", X.shape)
print("y shape:", y.shape)
print("Pixel range:", X.min(), "to", X.max())


In [ ]:
# Cell 12: Normalize pixel values and prepare CNN input

# Convert 0-255 pixels to 0-1
X = X / 255.0

# CNN expects: samples, height, width, channels
X = X[..., np.newaxis]

print("X shape:", X.shape)
print("Pixel range:", X.min(), "to", X.max())


In [ ]:
# Cell 13: Train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training images:", len(X_train))
print("Testing images:", len(X_test))

print("\nTraining class distribution:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nTesting class distribution:")
print(pd.Series(y_test).value_counts().sort_index())


In [ ]:
# Cell 14: Build the CNN
model = models.Sequential([
    # Change shape from (IMAGE_SIZE, IMAGE_SIZE, 1) to (MAX_HEIGHT, MAX_WIDTH, 1)
    layers.Input(shape=(MAX_HEIGHT, MAX_WIDTH, 1)), 
    
    # Data augmentation
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.08, 0.08),
    
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(9, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

In [ ]:
# Cell 15: Train the CNN

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.20,
    epochs=50,
    batch_size=8,
    callbacks=[early_stopping],
    verbose=1
)


In [ ]:
# Cell 16: Plot training history

plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CNN Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Cell 17: Evaluate on the test set

test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy * 100:.2f}%")


In [ ]:
# Cell 18: Classification report

predictions = model.predict(X_test, verbose=0)
y_pred = np.argmax(predictions, axis=1)

target_names = [CLASS_NAMES[i] for i in range(1, 10)]

print(classification_report(
    y_test,
    y_pred,
    target_names=target_names,
    zero_division=0
))


In [ ]:
# Cell 19: Confusion matrix

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 7))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks(range(9), target_names)
plt.yticks(range(9), target_names)

for i in range(9):
    for j in range(9):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.show()


In [ ]:
# Cell 20: Display test predictions

fig, axes = plt.subplots(3, 3, figsize=(9, 9))

for ax, image, true_label, prediction in zip(
    axes.ravel(),
    X_test[:9],
    y_test[:9],
    predictions[:9]
):
    predicted_label = np.argmax(prediction)

    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(
        f"True: {CLASS_NAMES[true_label + 1]} | "
        f"Pred: {CLASS_NAMES[predicted_label + 1]}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Cell 21: Save the trained model

MODEL_PATH = PROJECT_ROOT / "models" / "amharic_geez_digit_cnn.keras"

model.save(MODEL_PATH)

print("Model saved as:", MODEL_PATH)
